#### READ `CUSTOMERS` FROM VOLUME IN SPARK DATAFRAME
1. EXTRACT FILE_NAME AS METADATA
2. ADD `LOAD_TIMESTAMP` AS AUDIT COLUMN

In [0]:
from pyspark.sql.functions import col, current_timestamp

customer_df = (spark.read.format('json').load('/Volumes/gizmo/landing/operational_data/customers/')
                    .withColumn("file_name", col("_metadata"))
                    .withColumn("load_timestamp", current_timestamp())
)
display(customer_df.limit(5))

#### SELECT REQUIRED COLUMNS

In [0]:
from pyspark.sql.functions import to_date, to_timestamp

customer_file_dtls_df = customer_df.select("customer_id", "customer_name",
                                           to_date("date_of_birth",'yyyy-MM-dd').alias("date_of_birth"),
                                            "email", "member_since", "telephone", 
                                            to_timestamp("created_timestamp",'yyyy-MM-dd HH:mm:ss').alias("created_timestamp"),
                                           "file_name.file_name","file_name.file_path", "load_timestamp")
display(customer_file_dtls_df.limit(5))

#### WRITE `CUSTOMERS` INTO `DELTA FORMAT`

In [0]:
# customer_file_dtls_df.write.mode('overwrite').format('delta').saveAsTable("gizmo.bronze.customers_delta")
customer_file_dtls_df.writeTo("gizmo.bronze.customers_delta").createOrReplace()

#### VALIDATE AND QUERY THE TABLE `GIZMO.BRONZE.CUSTOMERS_DELTA`

In [0]:
%sql
SELECT * FROM GIZMO.BRONZE.CUSTOMERS_DELTA;

In [0]:
dbutils.notebook.exit("CUSTOMERS HAS BEEN LOADED SUCCESSFULLY INTO GIZMO.BRONZE.CUSTOMERS_DELTA")